In [1]:
from pathlib import Path
import random
import shutil
import pandas as pd
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

random.seed(42)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")

RAW_DIR = DATA_DIR / "raw"

FAIRFACE_DIR = RAW_DIR / "fairface"
CELEBA_DIR = RAW_DIR / "celeba"
DROWSINESS_DIR = RAW_DIR / "driver_drowsiness"
DISTRACTION_DIR = RAW_DIR / "driver_distraction"

for folder in [

    FAIRFACE_DIR,
    CELEBA_DIR,
    DROWSINESS_DIR,
    DISTRACTION_DIR

]:

    (folder/"images").mkdir(
        parents=True,
        exist_ok=True
    )

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/workspaces/DBA-ai-trust-score-framework-Interim")

SAMPLE_DIR = PROJECT_DIR / "data" / "samples"

FAIRFACE_DIR = SAMPLE_DIR / "fairface"
CELEBA_DIR = SAMPLE_DIR / "celeba"
DROWSINESS_DIR = SAMPLE_DIR / "driver_drowsiness"

for folder in [
    FAIRFACE_DIR,
    CELEBA_DIR,
    DROWSINESS_DIR,
    DISTRACTION_DIR
]:

    (folder/"images").mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ Folder structure created.")

✅ Folder structure created.


In [4]:
import os
from huggingface_hub import login

token = os.getenv("HF_TOKEN")

if token:
    login(token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


_________________Save Function______________________

In [5]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

def save_dataset_sample(dataset, save_dir, dataset_name):

    save_dir = Path(save_dir)
    image_dir = save_dir / "images"
    image_dir.mkdir(parents=True, exist_ok=True)

    metadata = []

    for idx, sample in enumerate(tqdm(dataset)):

        # Detect image field
        if "image" in sample:
            image = sample["image"]

        elif "Image_data" in sample:
            image = sample["Image_data"]["file"]

        else:
            raise ValueError("Image column not found.")

        filename = f"{dataset_name}_{idx:05d}.jpg"

        image.save(image_dir / filename)

        row = {}

        for k, v in sample.items():

            if k not in ["image", "Image_data"]:

                row[k] = v

        row["file_name"] = filename

        metadata.append(row)

    metadata = pd.DataFrame(metadata)

    metadata.to_csv(
        save_dir / "metadata.csv",
        index=False
    )

    print(f"Saved {len(metadata)} images")

    return metadata

__________________________ Stratified FAIRFACE DATA LOADING__________________________

In [15]:
from datasets import load_dataset
import pandas as pd

fairface = load_dataset(
    "HuggingFaceM4/FairFace",
    "0.25",
    split="train"
)

meta = []

for idx in range(len(fairface)):

    s = fairface[idx]

    meta.append({
        "idx": idx,
        "gender": s["gender"],
        "race": s["race"],
        "age": s["age"]
    })

meta = pd.DataFrame(meta)

sample_index = (
    meta
    .groupby(["gender","race","age"], group_keys=False)
    .apply(lambda x: x.sample(frac=0.02, random_state=42))
)

sample_index = sample_index.sample(
    n=min(1000, len(sample_index)),
    random_state=42
)

fairface_sample = [fairface[i] for i in sample_index.idx]

fairface_metadata = save_dataset_sample(
    fairface_sample,
    FAIRFACE_DIR,
    "fairface"
)

100%|██████████| 1000/1000 [00:00<00:00, 1595.15it/s]

Saved 1000 images


_____________________________CelebA DATA LOADING_________________________________

In [14]:
from datasets import load_dataset

celeba = load_dataset(
    "student/celebA",
    split="train[:5000]"      # Load only first 5000 images
)

celeba_sample = (
    celeba
    .shuffle(seed=42)
    .select(range(400))
)

celeba_metadata = save_dataset_sample(
    celeba_sample,
    CELEBA_DIR,
    "celeba"
)

100%|██████████| 400/400 [17:33<00:00,  2.63s/it]

Saved 400 images


_______________________________DRIVER DROWSINESS DATA LOADING____________________________

In [15]:
driver = load_dataset(
    "MichalMlodawski/closed-open-eyes",
    split="train"
)

meta = []

for idx in range(len(driver)):

    s = driver[idx]

    meta.append({
        "idx": idx,
        "Label": s["Label"]
    })

meta = pd.DataFrame(meta)

sample_index = (
    meta
    .groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(n=min(500,len(x)), random_state=42))
)

driver_sample = [driver[i] for i in sample_index.idx]

driver_metadata = save_dataset_sample(
    driver_sample,
    DROWSINESS_DIR,
    "driver_drowsiness"
)


100%|██████████| 1000/1000 [00:01<00:00, 591.40it/s]


Saved 1000 images


In [ ]:
print("FairFace :", len(fairface_metadata))
print("CelebA :", len(celeba_metadata))
print("Drowsiness :", len(driver_metadata))